# Кредитный скоринг физических лиц
Учебный проект LendingClub. **Ограничение:** `Charged Off` — прокси списания, не доказательство просрочки 90+ дней. Статусы завершённых кредитов могут приводить к смещению выборки. Все расчёты прибыли условные. Не используйте для реальных кредитных решений.

Ноутбуки запускаются последовательно 01 → 02 → 03 → 04. Каталог `artifacts/` сохраняет промежуточные результаты. Один раз разместите архив `accepted_2007_to_2018Q4.csv.gz` в текущем каталоге Colab или укажите путь в ячейке ниже. При новом сеансе Colab локальные файлы пропадают: используйте Google Drive или загрузите готовые промежуточные файлы.

Зависимости: `pip install pandas numpy matplotlib seaborn scikit-learn lightgbm optbinning shap joblib pyyaml`.

## 02. Инженерия признаков, WoE, логистическая регрессия

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
os.makedirs('artifacts', exist_ok=True)
def create_features(data):
    d=data.copy()
    num=['annual_inc','loan_amnt','installment','dti','fico_range_low','fico_range_high','revol_bal','open_acc','total_acc','delinq_2yrs']
    for c in num: d[c]=pd.to_numeric(d[c], errors='coerce')
    d['income_monthly']=d['annual_inc']/12
    d['loan_to_income']=d['loan_amnt']/(d['annual_inc']+1)
    d['payment_to_income']=d['installment']/(d['income_monthly']+1)
    d['debt_to_income']=d['dti']/100
    d['fico_average']=(d['fico_range_low']+d['fico_range_high'])/2
    d['fico_spread']=d['fico_range_high']-d['fico_range_low']
    d['credit_per_account']=d['revol_bal']/(d['open_acc']+1)
    d['account_activity']=d['open_acc']/(d['total_acc']+1)
    d['delinq_per_account']=d['delinq_2yrs']/(d['total_acc']+1)
    d['revolving_utilization']=pd.to_numeric(d['revol_util'].astype(str).str.replace('%','',regex=False),errors='coerce')/100
    return d
from sklearn.metrics import roc_auc_score, roc_curve
def metrics(y,p):
    auc=roc_auc_score(y,p);fpr,tpr,_=roc_curve(y,p)
    return {'ROC-AUC':auc,'Gini':2*auc-1,'KS':float(np.max(tpr-fpr))}
def profit(y,p,loan,threshold,good_return=.15,lgd=.45):
    y=np.asarray(y); p=np.asarray(p); loan=np.asarray(loan)
    return float(np.where(y==0,good_return*loan,-lgd*loan)[p<threshold].sum())

from optbinning import BinningProcess
from sklearn.linear_model import LogisticRegression
train=create_features(pd.read_pickle('artifacts/train.pkl'))
valid=create_features(pd.read_pickle('artifacts/valid.pkl'))
base_features=['loan_amnt', 'installment', 'annual_inc', 'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high', 'open_acc', 'pub_rec', 'revol_bal', 'total_acc']
new_features=['income_monthly', 'loan_to_income', 'payment_to_income', 'debt_to_income', 'fico_average', 'fico_spread', 'credit_per_account', 'account_activity', 'delinq_per_account', 'revolving_utilization']
model_features=base_features+new_features
X_train=train[model_features].replace([np.inf,-np.inf],np.nan)
X_valid=valid[model_features].replace([np.inf,-np.inf],np.nan)
y_train=train.target;y_valid=valid.target
binning=BinningProcess(variable_names=model_features)
binning.fit(X_train,y_train)
Xt=binning.transform(X_train,metric='woe')
Xv=binning.transform(X_valid,metric='woe')
base_model=LogisticRegression(max_iter=1000,class_weight='balanced',random_state=42)
base_model.fit(Xt,y_train)
p_base=base_model.predict_proba(Xv)[:,1]
print(metrics(y_valid,p_base))
joblib.dump({'binning':binning,'features':model_features,'base_model':base_model},'artifacts/baseline.pkl')
np.save('artifacts/p_base_valid.npy',p_base)


In [ ]:
from sklearn.metrics import RocCurveDisplay
RocCurveDisplay.from_predictions(y_valid,p_base)
plt.title('Logistic Regression — validation');plt.show()
print('Создано признаков:',len(new_features))
